# Static POD models for NOAA data

Dataset: Weekly mean sea-surface temperature as given by the NOAA Optimum Interpolation SST V2 dataset [[1]](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html)

Models: Static SHRED & SDN from [[2]](https://royalsocietypublishing.org/rspa/article/480/2298/20240054/66770/Sensing-with-shallow-recurrent-decoder) [[3]](https://github.com/Jan-Williams/pyshred/tree/main)

In [ ]:
# Standard imports
import json
import os
import random

# Third-party imports
import numpy as np
import pandas as pd
from scipy.interpolate import RBFInterpolator

# Local imports
from helper_functions import noaa_data, ofam3_data
from helper_functions.plotting_functions import plot_sst_map
from helper_functions.pod import create_pod_test_data, qr_place, svd_basis
from helper_functions.preprocess_SHRED import (
    convert_to_anomaly,
    split_ordered_data,
)
from helper_functions.saving import JsonEncoder

np.random.seed(42)
random.seed(42)

## Set up

In [ ]:
dataset = "NOAA"  # "OFAM3" or "NOAA"
sensor_placement = "QR"  # "QR" or "random"
num_sensors = 3  # 3 for NOAA experiments, 100 for OFAM3
lags = 52
anomaly = False
area = "NA"

# Storage
pod_name = f"{dataset}/{area}/{sensor_placement}/POD"
rbfi_name = f"{dataset}/{area}/{sensor_placement}/RBFI"

## Load data
From Mobile-Sensor SHRED paper [[4]](https://ieeexplore.ieee.org/document/10584544):

- Weekly mean SST from 1992 - 2019
- 1400 snapshots of a 180 x 360 grid, with landmass excluded leaves 44,219 spatial locations corresponding to the sea surface

In [ ]:
if dataset == "NOAA":
    data, lats, longs, dates = noaa_data.load_data()

    if area != "world":
        data, lats, longs = noaa_data.crop_to_area(data, lats, longs, area=area)

elif dataset == "OFAM3":
    data, dates, lats, longs = ofam3_data.load_files(file_path=f"Data/OFAM3/{area}/")

print(data.shape, len(longs), len(lats))

## Data preprocessing

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, train_dates, val_dates, test_dates = split_ordered_data(data, dates, all_dates=True)

# Plots for santity checks:
plot_index = 0
sst_df = pd.DataFrame({
    "Longitude": longs,
    "Latitude": lats,
    "SST": train_data[plot_index]
})
plot_sst_map(sst_df, cmap="plasma", title=f"Actual SST, Date: {train_dates[plot_index]}", area=area)

In [ ]:
if anomaly:
    train_data, val_data, test_data, bulk_mean = convert_to_anomaly((train_data, val_data, test_data))
    anomaly_df = pd.DataFrame({
            "Longitude": longs,
            "Latitude": lats,
            "SST": train_data[plot_index]
        })
    plot_sst_map(anomaly_df, cmap="viridis", title=f"SST Anomaly, Date: {train_dates[plot_index]}", area=area)

    bulk_df = pd.DataFrame({
        "Longitude": longs,
        "Latitude": lats,
        "SST": bulk_mean
    })
    plot_sst_map(bulk_df, cmap="plasma", title=f"Bulk mean SST, Dates: {train_dates[0]} to {train_dates[-1]}", area=area)

In [ ]:
print(train_data.shape)
print(val_data.shape)
print(test_data.shape)
print(len(train_data)+len(test_data)+len(val_data))

In [ ]:
if sensor_placement == "random":
    # Set sensor locations
    sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)
    sensor_coords=(longs[sensor_locations], lats[sensor_locations])
    print(f"Sensor location indexes will be: {sensor_locations}")

    # Set POD basis
    U_r = svd_basis(train_data, num_sensors)

elif sensor_placement == "QR":
    sensor_locations, U_r = qr_place(train_data, num_sensors)
    sensor_coords=(longs[sensor_locations], lats[sensor_locations])
    print(f"Sensor location indexes will be: {sensor_locations}")

else:
    raise ValueError("Incorrect placement of sensors specified")

In [ ]:
# Generate test set to match SHRED test set
# (use final sensor results for what each sequence would be if using SHRED)
test_input, ground_truth = create_pod_test_data(test_data, sensor_locations, test_lags=lags)
test_lag_dates = test_dates[np.array(range(len(test_dates)-lags)) + lags - 1]

In [ ]:
print(test_input.shape)
print(ground_truth.shape)

## Generate POD reconstructions

In [ ]:
C = np.zeros((num_sensors, len(lats)))
for i in range(num_sensors):
    C[i, sensor_locations[i]] = 1

A = C @ U_r

cond = np.linalg.cond(A)
if cond > 1e10:
    print(f"Warning: C @ U_r is ill-conditioned (cond={cond:.2e}) "
          f"for placement='{sensor_placement}'.")

coeffs, *_ = np.linalg.lstsq(A, test_input.T, rcond=None)
test_recons = (U_r @ coeffs).T

## Save POD results

In [ ]:
# Make experiment folder
directory = f"Reconstructions/Static/{pod_name}/"
os.makedirs(directory, exist_ok = True)

In [ ]:
np.save(f"Reconstructions/Static/{pod_name}/recons_n{num_sensors}_l{lags}", test_recons)
np.save(f"Reconstructions/Static/{pod_name}/truth_n{num_sensors}_l{lags}", ground_truth)

In [ ]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords),
    "test_lag_dates": list(test_lag_dates)
    }

if anomaly:
    metadata["bulk_mean"] = bulk_mean
with open(f"Reconstructions/Static/{pod_name}/metadata_n{num_sensors}_l{lags}.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, cls=JsonEncoder)

## Scipy interpolation

In [ ]:
points = np.array([[sensor_coords[0][i], sensor_coords[1][i]] for i in range(len(sensor_coords[0]))])
full_coords = np.column_stack([longs, lats])

In [ ]:
print(points.shape)
print(full_coords.shape)

In [ ]:
rbf_recons = np.zeros((len(test_input), len(lats)))
for j, data in enumerate(test_input):
    interpolator = RBFInterpolator(
        points,
        data,
        kernel='thin_plate_spline',
        smoothing=0.0,      # raise this (e.g. 1e-2–1e0) if sensors are noisy
    )
    rbf_recons[j] = interpolator(full_coords)

In [ ]:
print(test_input.shape)
print(ground_truth.shape)
print(rbf_recons.shape)

In [ ]:
df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": rbf_recons[3]
    })
plot_sst_map(df, area="NA", sensor_coords=sensor_coords, marker_size=15)

In [ ]:
directory = f"Reconstructions/Static/{rbfi_name}/"
os.makedirs(directory, exist_ok = True)
np.save(f"Reconstructions/Static/{rbfi_name}/recons_n{num_sensors}_l{lags}", rbf_recons)
np.save(f"Reconstructions/Static/{rbfi_name}/truth_n{num_sensors}_l{lags}", ground_truth)

In [ ]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords),
    "test_lag_dates": list(test_lag_dates)
    }

if anomaly:
    metadata["bulk_mean"] = bulk_mean
with open(f"Reconstructions/Static/{rbfi_name}/metadata_n{num_sensors}_l{lags}.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, cls=JsonEncoder)